# From Hodgkin–Huxley to a channelopathy: writing your own neuron in NESTML

**Goals**
1. Explore the classic Hodgkin–Huxley (HH) neuron built into NEST.
2. Re-create the same neuron in NESTML, generate code for it, and check that it behaves identically.
3. **Edit the NESTML file** to add a disease-causing sodium-channel mutation, regenerate, and compare wild type vs. mutant.

**Kernel:** on EBRAINS JupyterLab choose the most recent *EBRAINS-xx.xx* kernel (it ships NEST + NESTML).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nest
import pynestml
from pynestml.codegeneration.nest_code_generator_utils import NESTCodeGeneratorUtils

print("NEST  :", nest.__version__)
print("NESTML:", pynestml.__version__)

## Helper functions
`run_trace` records the membrane potential of one neuron driven by a current step.
`fi_curve` simulates many copies of a neuron, each with a different constant current `I_e`, and returns firing rates.

In [ ]:
def run_trace(model, amp=600., t_on=50., t_off=250., t_sim=300., params=None, record=("V_m",)):
    """Current step of `amp` pA from t_on to t_off. Returns times, dict of recorded variables, spike times."""
    nest.ResetKernel()
    nest.set_verbosity("M_WARNING")
    nest.resolution = 0.05
    n = nest.Create(model, params=params or {})
    stim = nest.Create("step_current_generator",
                       params={"amplitude_times": [t_on, t_off], "amplitude_values": [amp, 0.]})
    mm = nest.Create("multimeter", params={"record_from": list(record), "interval": 0.05})
    sr = nest.Create("spike_recorder")
    nest.Connect(stim, n); nest.Connect(mm, n); nest.Connect(n, sr)
    nest.Simulate(t_sim)
    ev = mm.get("events")
    return ev["times"], {k: ev[k] for k in record}, sr.get("events")["times"]


def fi_curve(model, currents, params=None, t_sim=1000., t_discard=200.):
    """Firing rate (Hz) for each constant current in `currents` (pA)."""
    nest.ResetKernel()
    nest.set_verbosity("M_WARNING")
    nest.resolution = 0.05
    pop = nest.Create(model, len(currents), params=params or {})
    pop.I_e = list(currents)
    sr = nest.Create("spike_recorder")
    nest.Connect(pop, sr)
    nest.Simulate(t_sim)
    ev = sr.get("events")
    ids = np.array(pop.tolist())
    return np.array([np.sum((ev["senders"] == i) & (ev["times"] > t_discard)) for i in ids]) \
        / ((t_sim - t_discard) / 1000.)

---
# Part 1 – Play with the built-in HH neuron (`hh_psc_alpha`)

In [ ]:
plt.figure(figsize=(10, 4))
for amp in [300., 700., 1500.]:
    t, rec, spk = run_trace("hh_psc_alpha", amp=amp)
    plt.plot(t, rec["V_m"], label=f"{amp:.0f} pA ({len(spk)} spikes)")
plt.xlabel("time (ms)"); plt.ylabel("V_m (mV)"); plt.legend(); plt.title("hh_psc_alpha, current steps 50–250 ms")
plt.show()

In [ ]:
currents = np.arange(0., 2001., 50.)
rate_builtin = fi_curve("hh_psc_alpha", currents)
plt.plot(currents, rate_builtin, "o-")
plt.xlabel("I_e (pA)"); plt.ylabel("rate (Hz)"); plt.title("f–I curve, built-in HH")
plt.show()

**Questions (Part 1)**
- What is the minimal current that produces repetitive firing (rheobase)? Is the onset of firing continuous (type I) or abrupt (type II)?
- Change `g_K` or `g_Na` via `params={...}` in `run_trace` and describe what happens.

---
# Part 2 – The same neuron written in NESTML

The next cell writes the model to the file **`hh_wt_neuron.nestml`** (you'll see it appear in the file browser on the left). Read it carefully: the `equations` block contains exactly the HH equations you know from the lecture.

In [ ]:
%%writefile hh_wt_neuron.nestml
# Classic Hodgkin-Huxley neuron (squid axon parameters, as in NEST's hh_psc_alpha)
# Stimulated by current only (I_e parameter or current generators via I_stim).
model hh_wt_neuron:

    state:
        V_m mV = -65 mV       # membrane potential
        Act_m real = 0.0529   # Na activation   (steady state at -65 mV)
        Act_h real = 0.5961   # Na inactivation (steady state at -65 mV)
        Act_n real = 0.3177   # K activation    (steady state at -65 mV)

    equations:
        # ---- ionic currents -------------------------------------------------
        inline I_Na pA = g_Na * Act_m**3 * Act_h * (V_m - E_Na)
        inline I_K  pA = g_K * Act_n**4 * (V_m - E_K)
        inline I_L  pA = g_L * (V_m - E_L)

        # ---- membrane equation ----------------------------------------------
        V_m' = (-(I_Na + I_K + I_L) + I_e + I_stim) / C_m

        # ---- gating kinetics (V in mV, rates in 1/ms) ------------------------
        inline alpha_m real = 0.1 * (V_m / (1 mV) + 40.) / (1. - exp(-(V_m / (1 mV) + 40.) / 10.))
        inline beta_m  real = 4. * exp(-(V_m / (1 mV) + 65.) / 18.)
        inline alpha_h real = 0.07 * exp(-(V_m / (1 mV) + 65.) / 20.)
        inline beta_h  real = 1. / (1. + exp(-(V_m / (1 mV) + 35.) / 10.))
        inline alpha_n real = 0.01 * (V_m / (1 mV) + 55.) / (1. - exp(-(V_m / (1 mV) + 55.) / 10.))
        inline beta_n  real = 0.125 * exp(-(V_m / (1 mV) + 65.) / 80.)

        Act_m' = (alpha_m * (1 - Act_m) - beta_m * Act_m) / (1 ms)
        Act_h' = (alpha_h * (1 - Act_h) - beta_h * Act_h) / (1 ms)
        Act_n' = (alpha_n * (1 - Act_n) - beta_n * Act_n) / (1 ms)

    parameters:
        C_m  pF = 100 pF
        g_Na nS = 12000 nS
        g_K  nS = 3600 nS
        g_L  nS = 30 nS
        E_Na mV = 50 mV
        E_K  mV = -77 mV
        E_L  mV = -54.402 mV
        I_e  pA = 0 pA        # constant injected current

    input:
        I_stim pA <- continuous

    output:
        spike

    update:
        V_m_old mV = V_m
        integrate_odes()
        # register a spike on upward crossing of 0 mV
        if V_m_old < 0 mV and V_m >= 0 mV:
            emit_spike()

In [ ]:
# Generate C++ code, compile it and load it into NEST (takes ~1 minute)
wt_module, wt_model = NESTCodeGeneratorUtils.generate_code_for(open("hh_wt_neuron.nestml").read())
nest.Install(wt_module)
print("Model available in NEST as:", wt_model)

**Sanity check:** the NESTML neuron must behave like the built-in one.

In [ ]:
t1, r1, _ = run_trace("hh_psc_alpha", amp=700.)
t2, r2, _ = run_trace(wt_model, amp=700.)
plt.figure(figsize=(10, 4))
plt.plot(t1, r1["V_m"], lw=3, alpha=0.5, label="built-in hh_psc_alpha")
plt.plot(t2, r2["V_m"], "k--", lw=1, label="NESTML hh_wt_neuron")
plt.xlabel("time (ms)"); plt.ylabel("V_m (mV)"); plt.legend(); plt.show()

---
# Part 3 – Your task: model a sodium channelopathy

Many inherited diseases (some epilepsies, paramyotonia congenita, long-QT syndrome type 3, hyperkalemic periodic paralysis) are caused by mutations in voltage-gated **Na⁺ channels** that impair **fast inactivation**: the mutant channel opens normally, but does not close again.

**Extremely simplified model:** a fraction `p_mut` of all Na⁺ channels is mutant and has *no h gate*:

$$I_{Na,WT} = (1-p_{mut})\,\bar g_{Na}\, m^3 h\,(V-E_{Na}) \qquad I_{Na,mut} = p_{mut}\,\bar g_{Na}\, m^3\,(V-E_{Na})$$

Run the next cell: it creates **`hh_mut_neuron.nestml`**, a copy of the wild-type file. Then **open `hh_mut_neuron.nestml` in the JupyterLab editor** (double-click it in the file browser) and:

1. Add a new parameter `p_mut real = 0.02` in the `parameters` block.
2. Multiply the existing `I_Na` by `(1 - p_mut)`.
3. Add a new inline equation `I_NaMut` for the mutant channels (it has no `Act_h`!).
4. Add `I_NaMut` to the membrane equation `V_m'`.
5. Save the file (Ctrl+S).

*Hint:* units must be consistent – `I_NaMut` is in `pA`, like `I_Na`. NESTML will tell you if they are not.

In [ ]:
# Create the file for students to edit (only run once – it would overwrite your edits!)
import os
if not os.path.exists("hh_mut_neuron.nestml"):
    src = open("hh_wt_neuron.nestml").read().replace("model hh_wt_neuron:", "model hh_mut_neuron:")
    open("hh_mut_neuron.nestml", "w").write(src)
    print("Created hh_mut_neuron.nestml – now open it and edit it!")
else:
    print("hh_mut_neuron.nestml already exists – not overwritten.")

In [ ]:
# After editing and saving the file: generate and load the mutant model
mut_module, mut_model = NESTCodeGeneratorUtils.generate_code_for(open("hh_mut_neuron.nestml").read())
nest.Install(mut_module)
print("Model available in NEST as:", mut_model)

---
# Part 4 – Wild type vs. mutant

Because `p_mut` is a *parameter*, we can change it from Python without regenerating code.

**Check 1:** with `p_mut = 0` the mutant model must be identical to the wild type. Why?

In [ ]:
amp = 600.
t, rw, sw = run_trace(wt_model, amp=amp)
plt.figure(figsize=(10, 4))
plt.plot(t, rw["V_m"], "k", lw=2, label=f"wild type ({len(sw)} spikes)")
for p in [0.0, 0.05, 0.2]:
    t, rm, sm = run_trace(mut_model, amp=amp, params={"p_mut": p})
    plt.plot(t, rm["V_m"], "--", label=f"p_mut = {p} ({len(sm)} spikes)")
plt.xlabel("time (ms)"); plt.ylabel("V_m (mV)"); plt.legend(); plt.title(f"Step current {amp:.0f} pA")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(currents, fi_curve(wt_model, currents), "k-", lw=3, label="wild type")
for p in [0.02, 0.05, 0.12, 0.2]:
    plt.plot(currents, fi_curve(mut_model, currents, params={"p_mut": p}), "o-", ms=3, label=f"p_mut = {p}")
plt.xlabel("I_e (pA)"); plt.ylabel("rate (Hz)"); plt.legend(); plt.title("f–I curves")
plt.show()

**Look inside the channels.** The mutant current is not recorded directly, but we can compute it from the recorded state variables.

In [ ]:
p = 0.05
t, r, _ = run_trace(mut_model, amp=600., params={"p_mut": p}, record=("V_m", "Act_m", "Act_h"))
g_Na, E_Na = 12000., 50.
I_wt  = (1 - p) * g_Na * r["Act_m"]**3 * r["Act_h"] * (r["V_m"] - E_Na)
I_mut = p * g_Na * r["Act_m"]**3 * (r["V_m"] - E_Na)
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 6))
ax[0].plot(t, r["V_m"]); ax[0].set_ylabel("V_m (mV)")
ax[1].plot(t, I_wt, label="I_Na wild-type channels"); ax[1].plot(t, I_mut, label="I_Na mutant channels")
ax[1].set_ylabel("current (pA)"); ax[1].set_xlabel("time (ms)"); ax[1].legend(); ax[1].set_xlim(100, 160)
plt.show()

**Questions (Part 4)**
1. How does the rheobase change with `p_mut`? Relate this to *hyperexcitability* (e.g. in epilepsy or myotonia).
2. What happens to spike amplitude and width?
3. For `p_mut = 0.2` the neuron fires once and then stays at about −15 mV. This is **depolarization block**. Explain it using the gating variables `m`, `h` and `n`. Which symptom of periodic paralysis could it explain?
4. Why is a *small* fraction of mutant channels enough to change the behaviour so much? (Look at the plot of the two currents between spikes.)

**Extensions**
- *K⁺ channel loss of function* (e.g. KCNA1, episodic ataxia): add a parameter scaling `g_K` for a fraction of dysfunctional K⁺ channels.
- *Shifted activation*: add a parameter `dV_m` that shifts the voltage dependence of `alpha_m`, `beta_m` (i.e. replace `V_m` by `V_m - dV_m`) and study gain vs. loss of function.
- Replace the non-inactivating mutation by *slowed* inactivation: divide `alpha_h` and `beta_h` by a factor for the mutant population (needs a second h variable!).